# 10 — Performance
**Goal:** Handle large files efficiently. Topics: dtype optimization, vectorization vs `.apply()`, chunking, memory profiling.

When your Adobe export grows to millions of rows and `.apply()` takes 40 seconds, these techniques bring it down to under 2.

In [1]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import pandas as pd
import numpy as np
import time
from pathlib import Path

Path('data/raw/large').mkdir(parents=True, exist_ok=True)

## 1 — Memory profiling
Always start by knowing how much memory your DataFrame uses.

In [2]:
# Generate a medium-sized dataset (~500k rows)
np.random.seed(0)
n = 500_000

channels  = ['organic', 'paid', 'email', 'direct', 'referral']
campaigns = ['brand', 'nonbrand', 'remarketing', 'display', 'none']
countries  = ['CL', 'PE', 'CO', 'MX', 'AR']

df = pd.DataFrame({
    'date':     pd.date_range('2023-01-01', periods=n, freq='1min'),
    'channel':  np.random.choice(channels, n),
    'campaign': np.random.choice(campaigns, n),
    'country':  np.random.choice(countries, n),
    'visits':   np.random.randint(1, 100, n),
    'orders':   np.random.randint(0, 5, n),
    'revenue':  np.random.uniform(0, 1000, n).round(2),
    'bounce':   np.random.uniform(0, 1, n).round(3),
})

print(f'Shape: {df.shape}')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print()
df.dtypes

Shape: (500000, 8)
Memory: 39.5 MB



date        datetime64[us]
channel                str
campaign               str
country                str
visits               int64
orders               int64
revenue            float64
bounce             float64
dtype: object

In [3]:
# Per-column memory breakdown
mem = df.memory_usage(deep=True).drop('Index')
mem_mb = (mem / 1e6).round(2).sort_values(ascending=False)
print(mem_mb.to_string())

campaign    7.5
channel     7.0
country     5.0
date        4.0
visits      4.0
orders      4.0
revenue     4.0
bounce      4.0


## 2 — Dtype optimization
The biggest win: convert `object` columns to `category` and downcast numeric types.

In [4]:
def optimize_dtypes(df):
    df = df.copy()

    # Low-cardinality string columns → category
    for col in df.select_dtypes(include='object').columns:
        n_unique = df[col].nunique()
        if n_unique / len(df) < 0.5:  # less than 50% unique values
            df[col] = df[col].astype('category')
            print(f'{col}: object → category ({n_unique} unique values)')

    # Downcast integers
    for col in df.select_dtypes(include='int64').columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')

    # Downcast floats
    for col in df.select_dtypes(include='float64').columns:
        df[col] = pd.to_numeric(df[col], downcast='float')

    return df


df_opt = optimize_dtypes(df)

before = df.memory_usage(deep=True).sum() / 1e6
after  = df_opt.memory_usage(deep=True).sum() / 1e6
print(f'\nMemory: {before:.1f} MB → {after:.1f} MB  ({(1 - after/before)*100:.0f}% reduction)')

channel: object → category (5 unique values)
campaign: object → category (5 unique values)
country: object → category (5 unique values)

Memory: 39.5 MB → 10.5 MB  (73% reduction)


/var/folders/3m/z0qp5nwj1vlccby390hqt0kc0000gn/T/ipykernel_9968/2602081391.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


In [5]:
# Key insight: category dtype stores an integer index + a lookup table
# 5 unique values in 500k rows = 5 strings stored once instead of 500k times
print('channel dtype:', df_opt['channel'].dtype)
print('channel categories:', df_opt['channel'].cat.categories.tolist())
print('channel codes (first 10):', df_opt['channel'].cat.codes[:10].tolist())

channel dtype: category
channel categories: ['direct', 'email', 'organic', 'paid', 'referral']
channel codes (first 10): [4, 2, 0, 0, 0, 3, 0, 1, 4, 2]


## 3 — Vectorization vs `.apply()`
`.apply()` runs Python-level loops. Vectorized operations use C-level numpy — 10-100x faster.

In [6]:
# Scenario: compute CVR and label channels as 'paid' or 'organic'

# --- SLOW: .apply() with Python function ---
def label_channel(ch):
    return 'paid' if ch in ['paid', 'email'] else 'organic'

t0 = time.perf_counter()
_ = df['channel'].apply(label_channel)
apply_time = time.perf_counter() - t0

# --- FAST: vectorized with np.where ---
t0 = time.perf_counter()
_ = np.where(df['channel'].isin(['paid', 'email']), 'paid', 'organic')
vec_time = time.perf_counter() - t0

print(f'.apply() time:    {apply_time:.3f}s')
print(f'np.where() time:  {vec_time:.3f}s')
print(f'Speedup: {apply_time / vec_time:.0f}x')

.apply() time:    0.099s
np.where() time:  0.019s
Speedup: 5x


In [7]:
# --- CVR calculation: avoid apply, use column arithmetic ---

# Slow
t0 = time.perf_counter()
_ = df.apply(lambda row: row['orders'] / row['visits'] if row['visits'] > 0 else 0, axis=1)
slow_time = time.perf_counter() - t0

# Fast
t0 = time.perf_counter()
_ = df['orders'] / df['visits'].replace(0, np.nan)
fast_time = time.perf_counter() - t0

print(f'apply(axis=1) time: {slow_time:.3f}s')
print(f'column / column:    {fast_time:.4f}s')
print(f'Speedup: {slow_time / fast_time:.0f}x')

print()
print('Rule: if you can express it with + - * / .str .dt .isin np.where → use that, never apply()')

apply(axis=1) time: 2.587s
column / column:    0.0015s
Speedup: 1753x

Rule: if you can express it with + - * / .str .dt .isin np.where → use that, never apply()


In [8]:
# When is apply() acceptable?
# - Complex conditional logic involving multiple columns that can't be expressed with vectorized ops
# - Calling external functions (APIs, regex with capture groups, etc.)
# - Small DataFrames (<10k rows) where performance doesn't matter

# Multi-condition example — use np.select instead of apply:
conditions = [
    df['channel'].isin(['paid']) & (df['orders'] > 2),
    df['channel'].isin(['organic']),
]
choices = ['high-value paid', 'organic']
_ = np.select(conditions, choices, default='other')

print('np.select is the vectorized equivalent of if/elif/else chains')

np.select is the vectorized equivalent of if/elif/else chains


## 4 — Reading large files efficiently

In [9]:
# Write a large CSV to test with
df.to_csv('data/raw/large/large_events.csv', index=False)
import os
size_mb = os.path.getsize('data/raw/large/large_events.csv') / 1e6
print(f'File size: {size_mb:.1f} MB')

File size: 27.8 MB


In [10]:
# Technique 1: only load the columns you need
cols_needed = ['date', 'channel', 'visits', 'orders', 'revenue']

t0 = time.perf_counter()
df_all = pd.read_csv('data/raw/large/large_events.csv')
all_time = time.perf_counter() - t0
all_mem = df_all.memory_usage(deep=True).sum() / 1e6

t0 = time.perf_counter()
df_slim = pd.read_csv('data/raw/large/large_events.csv', usecols=cols_needed)
slim_time = time.perf_counter() - t0
slim_mem = df_slim.memory_usage(deep=True).sum() / 1e6

print(f'All columns:    {all_time:.2f}s, {all_mem:.1f} MB')
print(f'usecols=5:      {slim_time:.2f}s, {slim_mem:.1f} MB')

All columns:    0.42s, 49.0 MB
usecols=5:      0.34s, 32.5 MB


In [11]:
# Technique 2: specify dtypes at read time (no post-processing needed)
dtype_map = {
    'channel':  'category',
    'campaign': 'category',
    'country':  'category',
    'visits':   'int16',
    'orders':   'int8',
    'revenue':  'float32',
    'bounce':   'float32',
}

df_typed = pd.read_csv(
    'data/raw/large/large_events.csv',
    dtype=dtype_map,
    parse_dates=['date'],
)

typed_mem = df_typed.memory_usage(deep=True).sum() / 1e6
print(f'Default dtypes: {all_mem:.1f} MB')
print(f'Specified dtypes: {typed_mem:.1f} MB')
print(f'Reduction: {(1 - typed_mem/all_mem)*100:.0f}%')

Default dtypes: 49.0 MB
Specified dtypes: 11.0 MB
Reduction: 78%


## 5 — Chunking: when the file doesn't fit in RAM

In [12]:
# Process a file in chunks — useful when file > available RAM
# Pattern: aggregate each chunk, then combine the chunk results

chunk_size = 100_000
results = []

t0 = time.perf_counter()
for chunk in pd.read_csv('data/raw/large/large_events.csv', chunksize=chunk_size, dtype=dtype_map):
    # Do the heavy work per chunk
    chunk_agg = chunk.groupby('channel').agg(
        visits  = ('visits',  'sum'),
        orders  = ('orders',  'sum'),
        revenue = ('revenue', 'sum'),
    )
    results.append(chunk_agg)

# Combine chunk results
final = pd.concat(results).groupby(level=0).sum()
final['cvr'] = final['orders'] / final['visits'] * 100
elapsed = time.perf_counter() - t0

print(f'Chunked processing time: {elapsed:.2f}s')
print(f'Peak memory per chunk: ~{chunk_size * 0.0002:.0f} MB')
print()
final.round(2)

Chunked processing time: 0.41s
Peak memory per chunk: ~20 MB



,visits,orders,revenue,cvr
channel,,,,
direct,5011429,200022,49958156.0,3.99
email,4995510,199680,49956808.0,4.00
organic,4994153,200016,49990540.0,4.01
paid,4956202,197998,49598604.0,3.99
referral,5037636,201323,50176308.0,4.00


## 6 — Parquet vs CSV

In [13]:
# Parquet is the right format for intermediate/clean data
# CSV is for raw exports and sharing with non-technical people

df_typed.to_parquet('data/raw/large/large_events.parquet', index=False)

csv_size = os.path.getsize('data/raw/large/large_events.csv') / 1e6
pq_size  = os.path.getsize('data/raw/large/large_events.parquet') / 1e6

t0 = time.perf_counter()
_ = pd.read_csv('data/raw/large/large_events.csv', dtype=dtype_map)
csv_read = time.perf_counter() - t0

t0 = time.perf_counter()
_ = pd.read_parquet('data/raw/large/large_events.parquet')
pq_read = time.perf_counter() - t0

print(f'CSV:     {csv_size:.1f} MB  |  read: {csv_read:.2f}s')
print(f'Parquet: {pq_size:.1f} MB  |  read: {pq_read:.2f}s')
print(f'Parquet is {csv_size/pq_size:.1f}x smaller and {csv_read/pq_read:.1f}x faster to read')

CSV:     27.8 MB  |  read: 0.35s
Parquet: 6.6 MB  |  read: 0.16s
Parquet is 4.2x smaller and 2.2x faster to read


## Summary — Performance checklist

| Technique | When to use | Typical gain |
|---|---|---|
| `dtype` map on `read_csv` | Always, for large files | 2–5x memory |
| `usecols=` | When you don't need all columns | proportional |
| `category` dtype | String columns with < 50% unique values | 5–20x per column |
| Vectorized ops instead of `.apply()` | Anytime you compute row-by-row | 10–100x speed |
| `chunksize=` | File doesn't fit in RAM | enables processing |
| Parquet for clean data | After any ETL step | 2–5x smaller, 2–10x faster read |

**The golden rule:** profile first with `df.memory_usage(deep=True)` — optimize the worst offenders only.